# 44. Five seeds of order_A, with the indicators attached

This is job 2 of three in the revised plasticity study. It is the main
evidence run and it is the long one.

## What this notebook does

It trains the same nine-task stream five times. The only thing that differs
between the five is the random seed, which is the number that decides how the
generator starts out and the order the images arrive in. Everything else is
held identical.

At the end of each run it unlearns task 0 on purpose. An unlearn means pushing
the generated weights of one task back toward random noise, so the model loses
that task. It costs 100 update steps, which is almost nothing next to nine
tasks of training, and it leaves a model whose damage is known and located.

At every boundary it measures the plasticity indicators. A boundary is a point
between two steps: boundary 0 is before any training, boundary 1 is after the
first task, and so on up to boundary 10 after the unlearn. That is 11
measurements per run.

## What it is evidence for

**Section 4.1, seed convergence.** The claim is that effective rank lands in
the same band whatever value it starts at, so most of its reported fall is
really a fact about the random start and not about learning. Five seeds is
what makes that claim checkable.

**Section 4.3, weight scale.** Five of its seven pairs of saved models come
from here: the model before the unlearn and the model after it.

## Before you run this

Notebook 43 must have finished and picked a learning rate. Put that value in
`LEARNING_RATE` in section 2. The notebook stops if you leave it unset.

## Where to run it

Either machine works. The sweep measured about 1.4 hours per nine-task run on
an L4, so all five seeds is close to 7 hours. RunPod will hold a session that
long; Colab may not.

You do not have to choose up front, because the loop is safe to start again
from the top. Everything a run needs lives in R2, and a seed that already
finished is skipped without training. Restarting costs one download per
finished seed, not one run.

So on Colab, the short way is to run a few seeds at a time. Set `SEEDS` in
section 2 to `(0, 1)`, let it finish, then come back later and set it to
`(2, 3)` and then `(4,)`. The records build up in R2 across sessions, and
section 7 reads whichever seeds are present. Leaving `SEEDS` at all five also
works; you just risk a disconnect part way.

On RunPod, leave `SEEDS` alone and run it once.

## The gate at the end

Section 7 checks that the five seeds finished within 3 percentage points of
each other on mean accuracy. If they did not, stop. Do not start job 3 on a
setting that failed its own target.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, time, shutil, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules

# Importing uncle twice in one session gives two copies of the module with two
# copies of its constants, which is a confusing way to fail much later.
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

# This notebook runs at the current main. The nine-task design and the
# ten-group class split were added after the earlier runs, so a pinned
# checkout would not contain them.
if IN_COLAB:
    REPO = Path("/content/path-plasticity")
    if not REPO.exists():
        subprocess.run(["git", "clone",
                        "https://github.com/sumitasthana/PATH.git", str(REPO)],
                       check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    # On RunPod, clone the repository yourself and start Jupyter inside it.
    # This walks up from the working directory to find the checkout.
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")

for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                        package], check=True)
sys.path.insert(0, str(REPO))

import torch
from uncle import Config, run
from uncle.config import study_defaults, PROBE_TASK
from uncle.storage import R2Store
from uncle.streams import build_tasks
from uncle.tasks import PARTITIONS
from uncle.plasticity_indicators import (
    StreamObserver, fixed_image_sets, indicator_key)

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                               text=True).strip()
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)
# Let the tensor cores do the matrix multiplies in TF32. TF32 keeps the range
# of a normal 32-bit float but drops some of the digits after the point. The
# loss is far smaller than the difference between two random seeds, and both
# the A100 and the L4 run several times faster this way. If no A100 is free
# and the run lands on an L4, this is the single setting that matters most.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
# Every image here is 64 by 64 and never changes size, so letting cuDNN spend
# the first few batches picking its fastest algorithm pays for itself.
torch.backends.cudnn.benchmark = True

if torch.cuda.is_available():
    memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("GPU:", torch.cuda.get_device_name(0), f"| {memory:.0f} GB")
    # Where the memory goes at the ninth task, which is the heaviest point:
    # the generator and its Adam state are about 2.1 GB, the eight protected
    # tasks hold one frozen weight set each at 94 MB, and the protection term
    # builds eight more with their gradients. The peak measured this way is
    # roughly 8 GB, so a 24 GB L4 fits. An L4 is slower than an A100, not
    # smaller: expect two to three times the wall clock, not an out-of-memory
    # error. The first run prints a projection so you can decide then.
    if memory < 20:
        print("Under 20 GB. The peak here is near 8 GB at the ninth task, so "
              "this should still fit. Watch the first run before trusting it.")
else:
    print("No GPU found. This notebook trains for hours. Do not start it on CPU.")

## 2. Settings

In [ ]:
# The learning rate notebook 43 chose. There is no default here on purpose: a
# guessed value would quietly undo the point of running the sweep first.
LEARNING_RATE = 5e-4          # notebook 43's winner, 13 November 2026
#
# It won by 0.53 points over 3e-4, which is about a sixth of the 3-point
# spread the gate below treats as the noise floor, so the two are tied as
# far as the sweep could tell. If the gate fails, rerun this notebook at
# 3e-4 before tuning epochs or beta: it needs no new sweep and it held the
# first task best of the five values tested.

if LEARNING_RATE is None:
    raise ValueError(
        "Set LEARNING_RATE to the value notebook 43 chose. It is in the "
        "printed table and in sweep_summary.json under 'chosen'.")

STUDY_ID = "plasticity_revised_v1"
BUCKET = "uncle-experiments"

# One ordering, five seeds. order_A is 0, 1, 2, 3, 4, 5, 6, 7, 8. The other
# two orderings are job 3, in notebook 45.
ORDER = "order_A"
SEEDS = (0, 1, 2, 3, 4)

# The task unlearned at the end of each run: the one learned first in this
# ordering. The earlier study unlearned its own first task, and job 3 does the
# same for its orderings, so the damage arm stays comparable across all seven
# runs.
FORGET_TASK = "0"

# Held fixed across all five runs, the same values the sweep used, so the seed
# is the only thing that differs between them.
#
#   epochs           5 per task, the setting the earlier runs used
#   batch_size       64, likewise
#   initialization   hyperfan_in, which every run of the earlier study used.
#                    "legacy" exists so old checkpoints keep their output
#                    scales, and is not wanted for a clean re-run.
EPOCHS = 5
BATCH_SIZE = 64
INITIALIZATION = "hyperfan_in"

# Images per fixed measurement set, and whether to include the satellite set.
IMAGES_PER_SET = 256
INCLUDE_EUROSAT = True

# Which boundary models to keep as files in R2. The indicator record already
# measures everything at every boundary, so a saved model is only needed for
# work the record cannot answer later: the rescaling control, which needs the
# model before the unlearn and the model after it.
#
# Boundary 9 is the end of the nine-task stream, boundary 10 is after the
# unlearn, and boundary 4 is a mid-stream reference. A checkpoint holds the
# whole generator, which is 134 million parameters, so each file is about
# 550 MB rather than the size of one task's weights. Three per run across five
# runs is roughly 8 GB. Widen this to range(11) for every boundary if you have
# the space; check the size of the first upload before you do.
KEEP_BOUNDARIES = (4, 9, 10)

DATA_ROOT = (Path("/content/data/tiny-imagenet-200") if IN_COLAB
             else REPO / "data/tiny-imagenet-200")
EUROSAT_ROOT = Path("/content/eurosat") if IN_COLAB else REPO / "data/eurosat"
OUTPUT_ROOT = Path("/content/job2") if IN_COLAB else REPO / "outputs/job2-seeds"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Nine learns, then one forget. `study_defaults` supplies the nine.
STREAM_REQUESTS = study_defaults(ORDER)["requests"]
ALL_REQUESTS = STREAM_REQUESTS + (("forget", FORGET_TASK),)

print("Study:", STUDY_ID, "| order:", ORDER, "| seeds:", SEEDS)
print("Stream:", " ".join(task for _, task in STREAM_REQUESTS))
print("Then unlearn task:", FORGET_TASK)
print("Held out as the probe group, never trained:", PROBE_TASK)
print("Learning rate:", LEARNING_RATE, "| epochs:", EPOCHS,
      "| batch:", BATCH_SIZE)
print("Boundaries measured per run: 0 to", len(ALL_REQUESTS))
print("Task-trainings in this job:", len(STREAM_REQUESTS) * len(SEEDS))

## 3. Connect to R2

R2 is the cloud bucket. It is what makes the split between RunPod and Colab
safe: this notebook writes its records and checkpoints there as it goes, and
the analysis notebooks read the same files from Colab afterwards.

The rule is simple. If a result is not in R2, it does not exist. A RunPod
instance that dies holding nine tasks of unsaved work costs real money to
repeat.

You need three secrets: `R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and
`R2_SECRET_ACCESS_KEY`. On Colab put them in the secrets panel on the left.
On RunPod set them as environment variables before starting Jupyter.

In [ ]:
def secret(name):
    """Read a credential from the environment, or from Colab's secret store."""
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                        secret("R2_ACCESS_KEY_ID"),
                        secret("R2_SECRET_ACCESS_KEY"))
store.progress = True

CACHE = OUTPUT_ROOT / "cache"
CACHE.mkdir(parents=True, exist_ok=True)

print("Connected to R2 bucket:", BUCKET)
print("Records go under: uncle/indicators/" + STUDY_ID + "/")

## 4. Build the image sets

This is the slow part before training starts. Indexing the dataset costs more
than the first optimizer steps do, so it is done once here and every run in
this notebook reuses it.

Three sets of pictures are measured at every boundary, and they answer
different questions.

- `in_stream` is held-out test images of the nine tasks the model learns. It
  says what happens to data the model is being trained on.
- `probe` is the tenth group of classes, which is never trained on. Same
  dataset, same kind of picture, classes the model has never seen.
- `eurosat` is satellite photographs. Nothing about them resembles the
  training stream.

A measure that moves on all three is describing the generator. A measure that
moves only on `in_stream` is describing what the model learned about those
particular classes.

The same 256 pictures are used at every boundary and in every run. If the
pictures moved between two measurements, a change in the reading could mean
the model changed or the pictures changed, and there would be no way to tell
which. The adapter plasticity study made exactly that mistake and had to redo
its analysis.

In [ ]:
from torch.utils.data import ConcatDataset

# Every task any run in this notebook needs, built in one call. The nine
# learned groups are the same in every ordering, only their order differs.
STREAM_TASKS = sorted({task for _, task in ALL_REQUESTS}, key=int)

tasks = build_tasks(root=DATA_ROOT,
                    partition_path=PARTITIONS[20],
                    partition_seed=42,
                    include=tuple(STREAM_TASKS + [PROBE_TASK]),
                    download=True)
print("Loaded Tiny ImageNet images for", len(tasks), "tasks")

sources = {
    "in_stream": ConcatDataset([tasks[name]["test"] for name in STREAM_TASKS]),
    "probe": tasks[PROBE_TASK]["test"],
}

if INCLUDE_EUROSAT:
    from torchvision.datasets import EuroSAT
    from torchvision.transforms import ToTensor
    # Already 64 by 64, which is the size the backbone expects.
    eurosat = EuroSAT(root=str(EUROSAT_ROOT), transform=ToTensor(),
                      download=True)
    assert eurosat[0][0].shape == (3, 64, 64), (
        f"Unexpected shape {eurosat[0][0].shape}")
    sources["eurosat"] = eurosat

# The draw is seeded inside the library, not here, so every run of every
# notebook in this study reads the same pictures.
image_sets = fixed_image_sets(sources, count=IMAGES_PER_SET)
for name, images in image_sets.items():
    print(f"{name:<10} {tuple(images.shape)}")

## 5. Build the five configurations

One settings object per seed. `study_defaults` supplies the dataset, the task
list, beta and the backbone, so the only difference between these five is the
seed.

Beta is the term that holds earlier tasks still while a new one is learned.
The methodology tunes it only if the learning-rate sweep alone misses its
target, so it stays at the paper's Tiny ImageNet value of 0.01 here.

In [ ]:
configs = {}
for seed in SEEDS:
    settings = study_defaults(ORDER)
    settings["requests"] = ALL_REQUESTS
    configs[seed] = Config(
        **settings,
        seed=seed,
        learning_rate=LEARNING_RATE,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        initialization=INITIALIZATION,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )

example = configs[SEEDS[0]]
CHANCE = 100 / example.classes_per_task
print("classes per task:", example.classes_per_task,
      "| chance accuracy:", f"{CHANCE:.0f}%")
print("backbone:", example.backbone, "| beta:", example.beta,
      "| burn-in steps for the unlearn:", example.burn_in)
print("requests:", " ".join(f"{action[0].upper()}{task}"
                            for action, task in example.requests))
print()
print("runs to do:")
for seed in SEEDS:
    print(f"  {ORDER}_seed{seed}")

## 6. Run the streams

Each run does the nine tasks, then the unlearn, and measures the indicators at
every boundary along the way.

Three things make an interrupted session cheap to recover.

The training state is saved after every request, and the run picks it up on a
second call. Re-running this cell after a disconnect continues at the task it
died on, not at the top.

The indicator record is uploaded to R2 after every boundary, not at the end. A
boundary already in the record is skipped rather than measured again.

The boundary models named in `KEEP_BOUNDARIES` are uploaded as they are
written, not at the end of the run.

**One limitation, stated plainly.** The within-task loss curve that notebook
36 recorded is not collected here. The library calls the observer at request
boundaries but has no per-update hook on this code path, so those rows would
be empty. Every boundary indicator the paper uses is collected. Only the
curve inside a task is missing, and nothing in the paper's claims rests on it.

In [ ]:
from datetime import datetime
from statistics import mean


def clock():
    """The wall clock, so a printed line can be placed in the day."""
    return datetime.now().strftime("%H:%M:%S")


def spell(seconds):
    """A duration as h:mm:ss. These run to hours, so minutes alone would not do."""
    seconds = int(max(seconds, 0))
    return f"{seconds // 3600}:{seconds // 60 % 60:02d}:{seconds % 60:02d}"


def run_one(label, config, order, seed, forget_task, requests):
    """Train one stream, unlearn one task, measure every boundary.

    `label` names the run and its output folder. `requests` is the nine learns
    followed by the one forget, so the last boundary is the damaged model.
    Returns a summary dictionary; the full record is the JSONL file in R2.
    """
    output = OUTPUT_ROOT / label
    output.mkdir(parents=True, exist_ok=True)

    log_key = indicator_key(STUDY_ID, order, seed)
    baseline_key = log_key.replace(".jsonl", ".baseline.pt")
    log_path = output / "indicators.jsonl"
    baseline_path = log_path.with_suffix(".baseline.pt")
    checkpoint_path = output / "checkpoint.pt"

    # The training checkpoint also lives in R2, under one key that is
    # overwritten after every request. The machine's own disk is not a safe
    # place for it: a RunPod pod that goes away takes its disk with it, and a
    # Colab runtime recycles. Without this copy a restarted run would find its
    # indicator record waiting in R2 and then retrain nine tasks from zero to
    # refill a record it already has.
    resume_key = (f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}"
                  f"/resume.pt")

    # An interrupted run picks up its own record. Both files come back or
    # neither: relative weight change is measured against boundary zero, and
    # the record without its baseline would have nothing to measure against.
    if store.info(log_key) is not None:
        if store.info(baseline_key) is None:
            raise RuntimeError(
                f"{log_key} is in R2 but {baseline_key} is missing. Boundary "
                "zero cannot be recreated once the generator has moved. "
                "Delete the record and start this run again.")
        shutil.copyfile(store.download(log_key, CACHE), log_path)
        shutil.copyfile(store.download(baseline_key, CACHE), baseline_path)
        done = [json.loads(line) for line
                in log_path.read_text(encoding="utf-8").splitlines()
                if line.strip()]
        print(f"  resuming: {len(done)} boundaries already measured, "
              f"up to t = {max(row['t'] for row in done)}")
    else:
        log_path.unlink(missing_ok=True)
        baseline_path.unlink(missing_ok=True)
        print("  starting fresh: no record in R2 yet")

    # Bring the training checkpoint back before the run starts. Only when
    # there is no local copy: a local file is the newer of the two, because it
    # is written first and uploaded after.
    if not checkpoint_path.exists() and store.info(resume_key) is not None:
        shutil.copyfile(store.download(resume_key, CACHE), checkpoint_path)
        print("  training checkpoint restored from R2")

    observer = StreamObserver(log_path, image_sets,
                              study=STUDY_ID, order=order, seed=seed)
    handle = {}

    def upload_record():
        """Push the record and its baseline to R2, replacing the last copy."""
        if log_path.exists():
            store.upload(log_path, log_key, overwrite=True)
        if baseline_path.exists():
            store.upload(baseline_path, baseline_key, overwrite=True)

    def remember(info):
        """Called before the first request, with the trainer the run built.

        Boundary zero is measured here: the fresh generator, before anything
        is learned. It is the value every later weight change is measured
        against, which is why it cannot be skipped. On a resume this is called
        a second time with `resumed_after`, and boundary zero is already in
        the record, so the observer returns from it without measuring again.
        """
        handle["uncle"] = info["uncle"]
        observer.boundary(info["uncle"], 0, None)
        upload_record()

    def measure(record):
        """Called after every finished request. Boundary t is index + 1."""
        t = record["index"] + 1
        row = observer.boundary(handle["uncle"], t, record["task"])
        upload_record()
        did = "learned " if record["action"] == "learn" else "UNLEARNED"
        accuracy = record["after"][record["task"]]
        extra = ""
        if row is not None:
            # Two headline numbers per boundary, so a long run says something
            # while it goes. The record holds every other measure.
            rank = row.get("effective_rank_in_stream")
            norm = row.get("norm_generator")
            travel = row.get("delta_w_generator")
            if rank is not None:
                extra += f" | eff.rank {rank:.1f}"
            if norm is not None:
                extra += f" | gen.norm {norm:.1f}"
            if travel is not None:
                extra += f" | moved {travel:.3f}"
        print(f"  [{clock()}] t={t:>2} {did} task {record['task']}: "
              f"{accuracy:5.1f}%{extra}")

    def keep_model(record, saved_path):
        """Send the saved checkpoint to R2 after every finished request.

        Two copies, for two different jobs. The resume copy is overwritten
        every time and is what a restarted run picks up. The boundary copies
        are separate files that stay, because the rescaling control needs the
        model from before the unlearn and the model from after it, and the
        resume copy will have moved past both by then.
        """
        t = record["index"] + 1
        saved_path = Path(saved_path)
        try:
            store.upload(saved_path, resume_key, overwrite=True)
        except Exception as failure:
            # A storage hiccup is not worth losing an hour of GPU time over.
            # The record for this boundary is already uploaded, and the next
            # request tries this upload again.
            print(f"        could not save the resume point: {failure}")
        if t not in KEEP_BOUNDARIES:
            return
        key = (f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}"
               f"/boundary_{t}.pt")
        try:
            store.upload(saved_path, key, overwrite=True)
            print(f"        kept the boundary {t} model in R2")
        except Exception as failure:
            print(f"        could not keep the boundary {t} model: {failure}")

    began = time.perf_counter()
    history = run(
        config,
        tasks=tasks,
        on_start=remember,
        on_request=measure,
        progress=True,
        checkpoint_path=checkpoint_path,
        resume=True,
        on_saved_checkpoint=keep_model,
    )
    took = time.perf_counter() - began

    # The record before the unlearn is the stream's own result. The last
    # record is after the unlearn, where one task is destroyed on purpose, so
    # reading mean accuracy there would mix damage into the gate.
    learned = [task for action, task in requests if action == "learn"]
    end_of_stream = history[len(learned) - 1]["after"]
    per_task = {task: end_of_stream[task] for task in learned}

    return {
        "run": label, "order": order, "seed": seed,
        "learning_rate": LEARNING_RATE,
        "accuracy_per_task": per_task,
        "mean_accuracy": mean(per_task.values()),
        "forgot_task": forget_task,
        "accuracy_of_forgotten_task": history[-1]["after"][forget_task],
        "boundaries_measured": len(requests) + 1,
        "wall_clock_seconds": took,
        "code_revision": HEAD,
    }

In [ ]:
results = {}

for index, seed in enumerate(SEEDS, start=1):
    print()
    print("=" * 70)
    print(f"[{index}/{len(SEEDS)}] {ORDER} seed {seed}   started {clock()}")
    print("=" * 70)

    row = run_one(label=f"{ORDER}_seed{seed}", config=configs[seed],
                  order=ORDER, seed=seed, forget_task=FORGET_TASK,
                  requests=ALL_REQUESTS)
    results[seed] = row

    print()
    print(f"  mean accuracy over the nine tasks: {row['mean_accuracy']:.2f}%")
    print(f"  task {FORGET_TASK} after the unlearn: "
          f"{row['accuracy_of_forgotten_task']:.2f}%  "
          f"(chance is {CHANCE:.0f}%)")
    print(f"  wall clock {spell(row['wall_clock_seconds'])}")

    # Printed after the first seed rather than at the end, so an unaffordable
    # job can be stopped here instead of after all five.
    if index == 1:
        each = row["wall_clock_seconds"]
        print()
        print("  Projection from this first run:")
        print(f"    rest of job 2, {len(SEEDS) - 1} seeds   "
              f"{spell(each * (len(SEEDS) - 1))}")
        print(f"    job 3, 2 orderings         {spell(each * 2)}")
        print("    Stop here if that is more than the budget allows.")

## 7. The result, and the gate

The methodology sets one target: five seeds, differing only in random start
and data order, must finish within 3 percentage points of each other on mean
accuracy over the nine tasks.

The reason is a floor. Every claim in the results compares two numbers read
from different runs. If two seeds of the identical setting can differ by 10
points, then a 10 point difference between two orderings means nothing. The
spread is what a difference has to clear before it counts for anything.

If the spread is wider than 3 points, stop. Do not run job 3.

In [ ]:
print(f"{'seed':>6}{'mean accuracy':>16}"
      f"{'task ' + FORGET_TASK + ' after unlearn':>26}{'hours':>8}")
for seed in SEEDS:
    if seed not in results:
        print(f"{seed:>6}{'not run':>16}")
        continue
    row = results[seed]
    print(f"{seed:>6}{row['mean_accuracy']:>15.2f}%"
          f"{row['accuracy_of_forgotten_task']:>25.2f}%"
          f"{row['wall_clock_seconds'] / 3600:>8.1f}")

spread = None
if len(results) == len(SEEDS):
    scores = [row["mean_accuracy"] for row in results.values()]
    spread = max(scores) - min(scores)
    print()
    print(f"Spread between the best and worst seed: {spread:.2f} "
          "percentage points.")
    if spread <= 3.0:
        print("PASS. The setting meets its target. Job 3 can start.")
    else:
        print("FAIL. The spread is wider than 3 points.")
        print("Do not start job 3. Tune further and repeat this notebook.")
        print("The methodology tunes epochs per task next, then beta.")

    # The damage arm assumes the unlearned task really is destroyed. If it is
    # not, Section 4.3 is reading a model that was never broken.
    worst = max(row["accuracy_of_forgotten_task"] for row in results.values())
    if worst > CHANCE + 5:
        print()
        print(f"Warning: the unlearned task did not reach chance "
              f"({CHANCE:.0f}%) in every run. The highest was {worst:.1f}%. "
              "Check this before using these models for Section 4.3.")

summary = {
    "study": STUDY_ID, "job": 2, "code_revision": HEAD,
    "order": ORDER, "seeds": list(SEEDS),
    "learning_rate": LEARNING_RATE,
    "classes_per_task": example.classes_per_task,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE,
    "beta": example.beta, "backbone": example.backbone,
    "initialization": INITIALIZATION,
    "forgot_task": FORGET_TASK,
    "kept_boundaries": list(KEEP_BOUNDARIES),
    "results": [results[seed] for seed in SEEDS if seed in results],
    "seed_spread": spread,
    "gate_passed": None if spread is None else spread <= 3.0,
}
summary_path = OUTPUT_ROOT / "job2_summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
store.upload(summary_path, f"uncle/indicators/{STUDY_ID}/job2_summary.json",
             overwrite=True)
print()
print("Written:", summary_path)
print("Uploaded to R2. Share this file and the table above.")

## 8. What comes next

If the gate passed, run notebook 45. That is job 3: the same nine tasks in two
different orders, one seed each, with the same unlearn appended. It is the
evidence for Section 4.2, the task-order claim.

Everything left after job 3 reads files and trains nothing, so it runs on
Colab:

- the rescaling control, four forward passes per pair of saved models
- the tables and figures, from the indicator records in R2

Nothing from this notebook has to be recomputed on another machine. The
records and the kept boundary models are in R2, and the later notebooks read
them from there.